[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# A Complete Project &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import copy
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' split, scaling, classifier, training loop, saved checkpoint and Keras model, for
the same made-up stations, rebuilt here from the same seeds so that each answer runs on its own.

In [2]:
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)


def as_tensors(rows):
    return (torch.from_numpy((features[rows] - mean) / std),
            torch.from_numpy(snow_on_may_1[rows]).unsqueeze(1))


X_train, y_train = as_tensors(train_rows)
X_valid, y_valid = as_tensors(valid_rows)
X_test, y_test = as_tensors(test_rows)


def make_classifier(hidden=16):
    if hidden == 0:
        return torch.nn.Linear(4, 1)
    return torch.nn.Sequential(torch.nn.Linear(4, hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, 1))


def fit(model, max_epochs=200, patience=15):
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_train, y_train), batch_size=32,
                                         shuffle=True, generator=torch.Generator().manual_seed(0))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    best_loss, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, max_epochs + 1):
        model.train()
        for features_batch, labels_batch in loader:
            optimizer.zero_grad()
            loss_fn(model(features_batch), labels_batch).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = loss_fn(model(X_valid), y_valid).item()
        if valid_loss < best_loss:
            best_loss, best_epoch, best_state = valid_loss, epoch, copy.deepcopy(model.state_dict())
        if epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)


def accuracy(model, X, y):
    with torch.no_grad():
        return ((model(X) > 0).float() == y).float().mean().item()


torch.manual_seed(0)
net = make_classifier()
fit(net)
torch.save({"model": net.state_dict(), "hidden": 16, "mean": torch.from_numpy(mean), "std": torch.from_numpy(std)},
           SCRATCH / "snow_classifier.pt")


def load_classifier(path):
    checkpoint = torch.load(path)
    model = make_classifier(checkpoint["hidden"])
    model.load_state_dict(checkpoint["model"])
    model.eval()

    def probability_of_snow(stations):
        scaled_stations = (torch.as_tensor(stations, dtype=torch.float32) - checkpoint["mean"]) / checkpoint["std"]
        with torch.no_grad():
            return torch.sigmoid(model(scaled_stations)).squeeze(1)

    return probability_of_snow


probability_of_snow = load_classifier(SCRATCH / "snow_classifier.pt")

normalization = keras.layers.Normalization()
normalization.adapt(features[train_rows])
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), normalization, keras.layers.Dense(16, activation="relu"),
                              keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss=keras.losses.BinaryCrossentropy(from_logits=True),
                  metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.0)])
history = net_keras.fit(features[train_rows], snow_on_may_1[train_rows],
                        validation_data=(features[valid_rows], snow_on_may_1[valid_rows]),
                        epochs=200, batch_size=32, verbose=0,
                        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                                 restore_best_weights=True)])

**1.** Train a PyTorch classifier with 32 hidden units with `make_classifier` and `fit`, from `torch.manual_seed(0)`, and print its validation accuracy beside `net`'s.

In [3]:
torch.manual_seed(0)
wide = make_classifier(32)
fit(wide)
print("32 units", round(accuracy(wide, X_valid, y_valid), 3), "| 16 units", round(accuracy(net, X_valid, y_valid), 3))

32 units 0.865 | 16 units 0.875


On the validation stations the network with 32 units gets 0.865 and the one with 16 units 0.875, a difference of 2 stations in 200, too few to choose between them. The test stations stay out of the comparison.

**2.** Print the share of test stations that `net` says have snow, beside the share that do.

In [4]:
with torch.no_grad():
    said_snow = (net(X_test) > 0).float().mean().item()
print("net says snow:", round(said_snow, 3), "| have snow:", round(y_test.mean().item(), 3))

net says snow: 0.395 | have snow: 0.465


The network says snow for 39.5% of the test stations, where 46.5% have it, so it calls more snow stations clear than it calls clear stations snow.

**3.** Use `probability_of_snow` to answer for three planned stations at once: (66.0, 14.0, 50.0, 2.0), (70.5, 25.0, 250.0, 60.0) and (62.0, 9.0, 1100.0, 90.0).

In [5]:
three = np.array([[66.0, 14.0, 50.0, 2.0], [70.5, 25.0, 250.0, 60.0], [62.0, 9.0, 1100.0, 90.0]], dtype="float32")
print(probability_of_snow(three).numpy().round(3))

[0.009 0.977 0.988]


The low station on the coast is almost certainly clear, at 0.009, and the inland station far to the north and the high one in the south almost certainly have snow, at 0.977 and 0.988. `probability_of_snow` takes several stations at once, one row each.

**4.** Count the test stations on which `net` and `net_keras` give different answers.

In [6]:
with torch.no_grad():
    net_says = (net(X_test) > 0).numpy().ravel()
keras_says = (net_keras(features[test_rows]).numpy() > 0).ravel()
print((net_says != keras_says).sum(), "of", len(test_rows))

0 of 200


The two networks give the same answer for all 200 test stations, which is why both score 0.82.

The answers saved a classifier in `scratch`, which this cell removes.

In [7]:
shutil.rmtree(SCRATCH)
print("scratch removed:", not SCRATCH.exists())

scratch removed: True


---

&#8592; **Back to:** [A Complete Project](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/15-a-complete-project.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
